# Chunking sweep: chunk size vs retrieval quality

Tests three chunk sizes (500, 1000, 1500 chars) on the same 484 ClinicalTrials.gov 
records to inform the production chunking choice. All three sizes use identical 
splitting parameters except chunk_size itself: same recursive splitter, same 200-char 
overlap, same separators, same min-chunk filter (100 chars). The 1000-char index is 
the deployed production build (loaded from disk); 500 and 1500 are rebuilt here.

Three evaluation passes:
1. Qualitative side-by-side on 7 probe queries (drug, biomarker, ADC vocabulary 
   mismatch, deeper-context queries, two adversarial OODs)
2. End-to-end RAG generation across 6 in-corpus + 4 adversarial queries × 3 sizes 
   = 30 runs
3. RAGAS-equivalent metrics (faithfulness, context precision) on the 18 in-corpus 
   runs, judged by the same Llama 3.3 70B used for generation

In [ ]:
import sys
import pickle
import time
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

sys.path.append("..")
from src.rag import (
    retrieve, generate, make_groq_client,
    EMBEDDING_MODEL, LLM_MODEL,
)

DATA_DIR = Path("../data")
SWEEP_DIR = DATA_DIR / "sweep"
SWEEP_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 80)

In [ ]:
# The deployed pipeline saved cleaned trials before chunking. Reuse them
# so the only variable across sizes is chunk_size itself.
with open(DATA_DIR / "trials.pkl", "rb") as f:
    trials = pickle.load(f)

print(f"Loaded {len(trials)} trials")
print(f"Sample fields: {list(trials[0].keys())}")

## Step A — Chunk at 500 and 1500 with production-matching parameters

Same `RecursiveCharacterTextSplitter` configuration as `notebooks/01_data_pipeline.ipynb`:
- `chunk_overlap=200` (not 100 as v1 used)
- `separators=["\n\n", "\n", ". ", " ", ""]` (5 separators including sentence boundaries)
- Min-chunk filter at 100 chars (drops list-fragment tails like "Exclusion Criteria:")

Only `chunk_size` varies.

In [ ]:
def chunk_trials(trials, chunk_size, overlap=200, min_chunk_len=100):
    """Re-chunk all trials with production-matching parameters.
    Returns list of dicts: {nct_id, title, text}, filtered to >= min_chunk_len."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )

    raw_chunks = []
    for trial in trials:
        for piece in splitter.split_text(trial["text"]):
            raw_chunks.append({
                "nct_id": trial["nct_id"],
                "title": trial["title"],
                "text": piece,
            })

    filtered = [c for c in raw_chunks if len(c["text"]) >= min_chunk_len]
    dropped = len(raw_chunks) - len(filtered)
    print(f"  size={chunk_size:>4} | raw={len(raw_chunks):>5,} | filtered={len(filtered):>5,} | dropped={dropped:>3}")
    return filtered


print("Chunking with production parameters (overlap=200, 5 separators, min=100):\n")
chunks_500  = chunk_trials(trials, chunk_size=500)
chunks_1500 = chunk_trials(trials, chunk_size=1500)

# Persist
with open(SWEEP_DIR / "chunks_500.pkl",  "wb") as f: pickle.dump(chunks_500,  f)
with open(SWEEP_DIR / "chunks_1500.pkl", "wb") as f: pickle.dump(chunks_1500, f)

print(f"\nFor reference: production 1000-char build = 3,264 chunks (loaded from disk later)")

In [ ]:
# Same model the production index was built with
print("Loading PubMedBERT (cached after first download)...")
t0 = time.time()
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded in {time.time()-t0:.0f}s | dim={model.get_sentence_embedding_dimension()} | device={model.device}")

# Embed both sizes
def embed_and_index(chunks, label):
    print(f"\nEncoding {label} ({len(chunks):,} chunks)...")
    t0 = time.time()
    embeddings = model.encode(
        [c["text"] for c in chunks],
        batch_size=32,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.0f}s ({len(chunks)/elapsed:.1f} chunks/sec)")

    idx = faiss.IndexFlatL2(embeddings.shape[1])
    idx.add(embeddings)
    return embeddings, idx


embeddings_500,  index_500  = embed_and_index(chunks_500,  "500-char")
embeddings_1500, index_1500 = embed_and_index(chunks_1500, "1500-char")

# Persist
faiss.write_index(index_500,  str(SWEEP_DIR / "faiss_500.index"))
faiss.write_index(index_1500, str(SWEEP_DIR / "faiss_1500.index"))
np.save(SWEEP_DIR / "embeddings_500.npy",  embeddings_500)
np.save(SWEEP_DIR / "embeddings_1500.npy", embeddings_1500)

In [ ]:
with open(DATA_DIR / "chunks.pkl", "rb") as f:
    chunks_1000 = pickle.load(f)
index_1000 = faiss.read_index(str(DATA_DIR / "faiss.index"))

print("All three indices loaded:")
print(f"  500:  {len(chunks_500):>5,} chunks | {index_500.ntotal:>5,} vectors")
print(f" 1000:  {len(chunks_1000):>5,} chunks | {index_1000.ntotal:>5,} vectors  (production)")
print(f" 1500:  {len(chunks_1500):>5,} chunks | {index_1500.ntotal:>5,} vectors")

## Step B — Qualitative comparison on 7 probe queries

Same query set as v1, same retrieval (`src.rag.retrieve`, k=5, dedup by NCT ID).
Tests whether the corrected sweep changes the qualitative findings: ADC ↔ T-DXd 
vocabulary mismatch, eligibility-numerical-constraint weakness, adversarial scoring 
band.

In [ ]:
PROBE_QUERIES = [
    {"query": "pembrolizumab in non-small cell lung cancer",
     "category": "drug + condition",
     "expect": "in-corpus, multi-concept; should rank well at all sizes"},

    {"query": "BRAF V600E mutation targeted therapy",
     "category": "biomarker baseline",
     "expect": "in-corpus, well-represented"},

    {"query": "antibody drug conjugate for HER2 positive cancer",
     "category": "vocabulary mismatch",
     "expect": "stress test surfaced T-DXd present in corpus but missed by retrieval; test which size recovers it"},

    {"query": "hazard ratio for overall survival in phase 3 trials",
     "category": "deeper context",
     "expect": "stats numbers usually deeper in trial body"},

    {"query": "trial eligibility criteria age 65 and older",
     "category": "deeper context",
     "expect": "tests if PubMedBERT under-weights numerical constraints"},

    {"query": "best Italian restaurant in Boston",
     "category": "adversarial OOD",
     "expect": "non-medical, scoring band reveals threshold inactivity"},

    {"query": "treatment for the common cold",
     "category": "adversarial in-domain vocab",
     "expect": "earlier sweep found 500-char leaked here on a misread of exclusion criteria"},
]

def retrieve_all_sizes(query, k=5):
    return {
        500:  retrieve(query, model, index_500,  chunks_500,  k=k),
        1000: retrieve(query, model, index_1000, chunks_1000, k=k),
        1500: retrieve(query, model, index_1500, chunks_1500, k=k),
    }

print(f"{len(PROBE_QUERIES)} probe queries loaded")

In [ ]:
all_qualitative = {}
for q_obj in PROBE_QUERIES:
    q = q_obj["query"]
    print("=" * 90)
    print(f"QUERY: {q}")
    print(f"  Category: {q_obj['category']}")
    print(f"  Expect:   {q_obj['expect']}")
    print("=" * 90)

    results_by_size = retrieve_all_sizes(q, k=5)
    all_qualitative[q] = results_by_size

    for size in [500, 1000, 1500]:
        rs = results_by_size[size]
        print(f"\n--- {size}-char ---  top-1 sim = {rs[0]['score']:.3f}")
        for r in rs[:3]:
            preview = r["text"][:200].replace("\n", " ")
            print(f"  #{r['rank']} [{r['nct_id']}] sim={r['score']:.3f}")
            print(f"      {preview}...")
    print()

In [ ]:
rows = []
for q_obj in PROBE_QUERIES:
    res = all_qualitative[q_obj["query"]]
    rows.append({
        "category":  q_obj["category"],
        "query":     q_obj["query"][:48],
        "top1_500":  round(res[500][0]["score"],  3),
        "top1_1000": round(res[1000][0]["score"], 3),
        "top1_1500": round(res[1500][0]["score"], 3),
        "nct_500":   res[500][0]["nct_id"],
        "nct_1000":  res[1000][0]["nct_id"],
        "nct_1500":  res[1500][0]["nct_id"],
    })

qual_summary = pd.DataFrame(rows)
qual_summary["all_agree_on_top"] = qual_summary.apply(
    lambda r: r["nct_500"] == r["nct_1000"] == r["nct_1500"], axis=1
)
qual_summary.to_csv(SWEEP_DIR / "qualitative_summary.csv", index=False)
qual_summary

## Step C — End-to-end RAG runs across all 12 queries × 3 sizes

6 in-corpus queries (the RAGAS evaluation set) + 4 adversarial queries, each through 
the full RAG pipeline at all 3 sizes. 30 runs total.

In [ ]:
EVAL_QUERIES = [
    "What trials use pembrolizumab in non-small cell lung cancer?",
    "Which trials target the BRAF V600E mutation?",
    "Are there trials studying antibody drug conjugates for HER2 positive cancer?",
    "What are reported hazard ratios for overall survival in phase 3 oncology trials?",
    "What are the eligibility criteria for trials enrolling patients age 65 and older?",
    "What trials use CAR-T cell therapy for leukemia or lymphoma?",
]
ADVERSARIAL_QUERIES = [
    "best Italian restaurant in Boston",
    "treatment for the common cold",
    "how do I file my taxes",
    "what is the capital of France",
]

INDICES_BY_SIZE = {500: index_500,  1000: index_1000, 1500: index_1500}
CHUNKS_BY_SIZE  = {500: chunks_500, 1000: chunks_1000, 1500: chunks_1500}

client = make_groq_client()

rag_runs = []
t0 = time.time()
for q in EVAL_QUERIES + ADVERSARIAL_QUERIES:
    for size in [500, 1000, 1500]:
        result = generate(q, client, model, INDICES_BY_SIZE[size], CHUNKS_BY_SIZE[size], k=5)
        rag_runs.append({
            "query":          q,
            "size":           size,
            "answer":         result["answer"],
            "contexts":       [r["text"] for r in result["sources"]],
            "context_ncts":   [r["nct_id"] for r in result["sources"]],
            "top_score":      result["top_score"],
            "refused_at":     result["refused_at"],
            "is_adversarial": q in ADVERSARIAL_QUERIES,
        })
        print(f"  {size:>4} | {q[:55]:55s} | refused={result['refused_at']}")

print(f"\nTotal: {len(rag_runs)} RAG runs in {time.time()-t0:.0f}s")
with open(SWEEP_DIR / "rag_runs.pkl", "wb") as f: pickle.dump(rag_runs, f)

In [ ]:
# Cell R1 — emergency save of in-memory state
import pickle
from pathlib import Path

SWEEP_DIR = Path("../data/sweep")

# Save whatever Cell 13 managed to complete
print(f"rag_runs entries completed: {len(rag_runs)}")
with open(SWEEP_DIR / "rag_runs_partial.pkl", "wb") as f:
    pickle.dump(rag_runs, f)

# Save the qualitative results (no Groq dependency, should be intact)
with open(SWEEP_DIR / "all_qualitative.pkl", "wb") as f:
    pickle.dump(all_qualitative, f)
qual_summary.to_csv(SWEEP_DIR / "qualitative_summary.csv", index=False)

# Confirm what made it
print(f"Saved partial state to {SWEEP_DIR}/")
print(f"  rag_runs_partial.pkl     ({len(rag_runs)} of 30 runs)")
print(f"  qualitative_summary.csv  ({len(qual_summary)} queries × 3 sizes)")
print(f"\nQualitative summary preview:")
print(qual_summary)

In [1]:
# Cell M1 — full state rehydration after restart
import sys, pickle, time, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

sys.path.append("..")
from src.rag import (
    retrieve, generate, make_groq_client,
    EMBEDDING_MODEL, LLM_MODEL,
)

DATA_DIR  = Path("../data")
SWEEP_DIR = DATA_DIR / "sweep"

# Load all three chunk sets
with open(SWEEP_DIR / "chunks_500.pkl",  "rb") as f: chunks_500  = pickle.load(f)
with open(SWEEP_DIR / "chunks_1500.pkl", "rb") as f: chunks_1500 = pickle.load(f)
with open(DATA_DIR  / "chunks.pkl",      "rb") as f: chunks_1000 = pickle.load(f)

# Load all three indices
index_500  = faiss.read_index(str(SWEEP_DIR / "faiss_500.index"))
index_1500 = faiss.read_index(str(SWEEP_DIR / "faiss_1500.index"))
index_1000 = faiss.read_index(str(DATA_DIR  / "faiss.index"))

# Load partial RAG runs and qualitative results
with open(SWEEP_DIR / "rag_runs_partial.pkl", "rb") as f: rag_runs = pickle.load(f)
with open(SWEEP_DIR / "all_qualitative.pkl",  "rb") as f: all_qualitative = pickle.load(f)

# Load model + client (needed for resumed RAG runs)
print("Loading PubMedBERT (cached)...")
t0 = time.time()
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded in {time.time()-t0:.0f}s | device={model.device}")
client = make_groq_client()

print(f"\nState rehydrated:")
print(f"  Chunks:    500={len(chunks_500):,}  1000={len(chunks_1000):,}  1500={len(chunks_1500):,}")
print(f"  Indices:   500={index_500.ntotal:,}  1000={index_1000.ntotal:,}  1500={index_1500.ntotal:,}")
print(f"  RAG runs:  {len(rag_runs)} of 30 saved from last night")
print(f"  Qualitative results: {len(all_qualitative)} queries")

Loading PubMedBERT (cached)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded in 2s | device=cpu

State rehydrated:
  Chunks:    500=6,855  1000=3,264  1500=2,138
  Indices:   500=6,855  1000=3,264  1500=2,138
  RAG runs:  23 of 30 saved from last night
  Qualitative results: 7 queries


In [ ]:
# Cell M2 — complete remaining RAG runs

EVAL_QUERIES = [
    "What trials use pembrolizumab in non-small cell lung cancer?",
    "Which trials target the BRAF V600E mutation?",
    "Are there trials studying antibody drug conjugates for HER2 positive cancer?",
    "What are reported hazard ratios for overall survival in phase 3 oncology trials?",
    "What are the eligibility criteria for trials enrolling patients age 65 and older?",
    "What trials use CAR-T cell therapy for leukemia or lymphoma?",
]
ADVERSARIAL_QUERIES = [
    "best Italian restaurant in Boston",
    "treatment for the common cold",
    "how do I file my taxes",
    "what is the capital of France",
]

INDICES_BY_SIZE = {500: index_500,  1000: index_1000, 1500: index_1500}
CHUNKS_BY_SIZE  = {500: chunks_500, 1000: chunks_1000, 1500: chunks_1500}

# Build the set of (query, size) pairs we already have
done = {(r["query"], r["size"]) for r in rag_runs}
print(f"Already complete: {len(done)} runs")

# Iterate the full 30-pair grid, skip what's already done
all_pairs = [(q, s) for q in EVAL_QUERIES + ADVERSARIAL_QUERIES for s in [500, 1000, 1500]]
todo = [(q, s) for (q, s) in all_pairs if (q, s) not in done]
print(f"Remaining:        {len(todo)} runs\n")

t0 = time.time()
for q, size in todo:
    result = generate(q, client, model, INDICES_BY_SIZE[size], CHUNKS_BY_SIZE[size], k=5)
    rag_runs.append({
        "query":          q,
        "size":           size,
        "answer":         result["answer"],
        "contexts":       [r["text"] for r in result["sources"]],
        "context_ncts":   [r["nct_id"] for r in result["sources"]],
        "top_score":      result["top_score"],
        "refused_at":     result["refused_at"],
        "is_adversarial": q in ADVERSARIAL_QUERIES,
    })
    print(f"  {size:>4} | {q[:55]:55s} | refused={result['refused_at']}")

print(f"\nCompleted {len(todo)} runs in {time.time()-t0:.0f}s")
print(f"Total rag_runs: {len(rag_runs)} of 30")

# Save the full set
with open(SWEEP_DIR / "rag_runs.pkl", "wb") as f:
    pickle.dump(rag_runs, f)
print(f"Saved: rag_runs.pkl")

In [12]:
# Cell M3 — judge helpers (LLM-as-judge for faithfulness + context precision)

REFUSAL_PHRASES = [
    "i don't have enough information",
    "outside the scope",
    "cannot answer",
    "do not have enough information",
]

def is_refusal(answer):
    return any(p in answer.lower() for p in REFUSAL_PHRASES)


def judge(prompt, max_tokens=300, temperature=0.0, max_retries=3):
    """Groq call with backoff on rate limits. Returns text or empty on failure."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            msg = str(e).lower()
            if "rate" in msg and attempt < max_retries - 1:
                wait = 2 ** attempt * 5
                print(f"    rate-limited, sleeping {wait}s")
                time.sleep(wait)
            else:
                print(f"    judge failed: {e}")
                return ""
    return ""


def parse_json_list(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match: return []
    try: return json.loads(match.group(0))
    except json.JSONDecodeError: return []


def parse_yes_no(text):
    t = text.strip().lower()
    if t.startswith("yes"): return 1
    if t.startswith("no"):  return 0
    if "yes" in t[:20]: return 1
    if "no"  in t[:20]: return 0
    return None


def faithfulness_score(answer, context_text):
    """% of atomic claims supported by retrieved context."""
    if is_refusal(answer):
        return 1.0, 0, 0
    decomp = (
        "Break the following answer into atomic factual claims. "
        "Return ONLY a JSON list of strings, no other text. "
        "If there are no factual claims, return [].\n\n"
        f"Answer: {answer}\n\nJSON:"
    )
    claims = parse_json_list(judge(decomp, max_tokens=400))
    if not claims:
        return 1.0, 0, 0
    supported = 0
    for claim in claims:
        v = parse_yes_no(judge(
            "Does the context support the claim? Answer ONLY 'yes' or 'no'.\n\n"
            f"Context:\n{context_text}\n\nClaim: {claim}\n\nAnswer:",
            max_tokens=10
        ))
        if v == 1: supported += 1
    return supported / len(claims), len(claims), supported


def context_precision_score(question, contexts):
    """% of retrieved chunks judged relevant to the question."""
    if not contexts: return 0.0
    relevant = 0
    for ctx in contexts:
        v = parse_yes_no(judge(
            "Is the document chunk relevant for answering the question? "
            "Answer ONLY 'yes' or 'no'.\n\n"
            f"Question: {question}\n\nChunk: {ctx[:1500]}\n\nAnswer:",
            max_tokens=10
        ))
        if v == 1: relevant += 1
    return relevant / len(contexts)


print("Helpers ready: is_refusal, judge, faithfulness_score, context_precision_score")

Helpers ready: is_refusal, judge, faithfulness_score, context_precision_score


In [ ]:
# Cell M4 — score 18 in-corpus runs with LLM judge

in_corpus_runs = [r for r in rag_runs if not r["is_adversarial"]]
print(f"Scoring {len(in_corpus_runs)} in-corpus runs (~6 min, ~30K Groq tokens)\n")

scored = []
t0 = time.time()
for i, run in enumerate(in_corpus_runs):
    ctx_text = "\n\n---\n\n".join(run["contexts"])
    f_score, n_claims, n_sup = faithfulness_score(run["answer"], ctx_text)
    prec = context_precision_score(run["query"], run["contexts"])
    scored.append({
        **run,
        "faithfulness":      round(f_score, 3),
        "n_claims":          n_claims,
        "n_supported":       n_sup,
        "context_precision": round(prec, 3),
    })
    elapsed = time.time() - t0
    print(f"  [{i+1:2d}/18] size={run['size']:>4}  "
          f"faith={f_score:.2f} ({n_sup}/{n_claims})  "
          f"prec={prec:.2f}  "
          f"elapsed={elapsed:.0f}s  | {run['query'][:50]}")

print(f"\nDone in {time.time()-t0:.0f}s")
with open(SWEEP_DIR / "scored.pkl", "wb") as f: pickle.dump(scored, f)
print(f"Saved: scored.pkl")

In [ ]:
# Cell M5 — figure out why 11 of 18 runs have n_claims=0

print(f"Total scored: {len(scored)}")
zero_claim_runs = [s for s in scored if s["n_claims"] == 0]
print(f"Runs with n_claims=0: {len(zero_claim_runs)}\n")

# For each zero-claim run: was the answer a refusal? An empty string? Something else?
for s in zero_claim_runs:
    is_ref = is_refusal(s["answer"])
    answer_preview = s["answer"][:200].replace("\n", " ")
    answer_len = len(s["answer"])
    print(f"size={s['size']:>4} | is_refusal={is_ref} | answer_len={answer_len}")
    print(f"  Q: {s['query'][:70]}")
    print(f"  A: {answer_preview}")
    print()

In [ ]:
# Cell M6 — corrected summary, separating refusals from answered runs

df = pd.DataFrame(scored)
in_corpus = df[~df["is_adversarial"]].copy()
in_corpus["was_refused"] = in_corpus["answer"].apply(is_refusal)

# Per-size breakdown
rows = []
for size in [500, 1000, 1500]:
    sub = in_corpus[in_corpus["size"] == size]
    answered = sub[~sub["was_refused"]]
    rows.append({
        "size": size,
        "n_total":          len(sub),
        "n_answered":       len(answered),
        "n_refused":        sub["was_refused"].sum(),
        "refusal_rate":     round(sub["was_refused"].mean(), 3),
        "faith_when_answered":  round(answered["faithfulness"].mean(), 3) if len(answered) else None,
        "ctx_precision_all":    round(sub["context_precision"].mean(), 3),
        "mean_claims_when_answered": round(answered["n_claims"].mean(), 1) if len(answered) else None,
    })
in_summary = pd.DataFrame(rows).set_index("size")

# Adversarial side
adv = pd.DataFrame([r for r in rag_runs if r["is_adversarial"]])
adv["is_refusal"] = adv["answer"].apply(is_refusal)
adv_summary = adv.groupby("size").agg(
    adv_refusal_rate=("is_refusal", "mean"),
    adv_mean_top_score=("top_score", "mean"),
).round(3)

print("=" * 80)
print("IN-CORPUS RESULTS (6 queries × 3 sizes)")
print("=" * 80)
print(in_summary)
print("\n" + "=" * 80)
print("ADVERSARIAL RESULTS (4 queries × 3 sizes)")
print("=" * 80)
print(adv_summary)

# Per-query detail — which queries answered vs refused at each size
print("\n" + "=" * 80)
print("PER-QUERY OUTCOMES (in-corpus)")
print("=" * 80)
pivot = in_corpus.pivot_table(
    index="query",
    columns="size",
    values="was_refused",
    aggfunc="first",
).rename(columns={500: "500", 1000: "1000", 1500: "1500"})
pivot.columns = [f"size_{c}_refused" for c in pivot.columns]
print(pivot)

# Save everything
in_summary.to_csv(SWEEP_DIR / "in_corpus_summary.csv")
adv_summary.to_csv(SWEEP_DIR / "adversarial_summary.csv")
pivot.to_csv(SWEEP_DIR / "per_query_outcomes.csv")
print(f"\nSaved: in_corpus_summary.csv, adversarial_summary.csv, per_query_outcomes.csv")

In [ ]:
# Cell M7 — inspect the ADC query in detail across sizes
adc_query = "Are there trials studying antibody drug conjugates for HER2 positive cancer?"
adc_runs = [r for r in rag_runs if r["query"] == adc_query]

for run in sorted(adc_runs, key=lambda r: r["size"]):
    print("=" * 80)
    print(f"SIZE: {run['size']} | refused: {is_refusal(run['answer'])}")
    print(f"NCTs retrieved: {run['context_ncts']}")
    print(f"\nANSWER:\n{run['answer']}")
    print()

## Step D — Reranker comparison (chunk size = 1000, rerank on vs off)

Holding chunk size at the production value (1000 chars), measure the impact of adding a cross-encoder reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to retrieval. The reranker scores all 20 FAISS candidates before dedup, so the cross-encoder picks which chunk per trial survives — not just the best FAISS chunk.

Reuses the same 6 in-corpus queries from Step C for an apples-to-apples comparison. Headline question: does reranking improve context precision and/or reduce refusal rate?

In [3]:
# Cell D1 — load cross-encoder reranker
from src.rag import load_reranker

print("Loading cross-encoder reranker (~90MB, cached after first download)...")
t0 = time.time()
reranker = load_reranker()
print(f"Loaded in {time.time()-t0:.0f}s | {type(reranker).__name__}")

Loading cross-encoder reranker (~90MB, cached after first download)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded in 2s | CrossEncoder


In [4]:
# Cell D2 — re-run 6 in-corpus queries at chunk size 1000, with reranker on vs off
# 12 runs total: 6 queries × 2 configs (no_rerank, with_rerank)

rerank_runs = []
t0 = time.time()
for q in EVAL_QUERIES:
    for use_rerank in [False, True]:
        rr = reranker if use_rerank else None
        result = generate(q, client, model, index_1000, chunks_1000, k=5, reranker=rr)
        rerank_runs.append({
            "query":        q,
            "size":         1000,
            "use_reranker": use_rerank,
            "answer":       result["answer"],
            "contexts":     [r["text"] for r in result["sources"]],
            "context_ncts": [r["nct_id"] for r in result["sources"]],
            "top_score":    result["top_score"],
            "refused_at":   result["refused_at"],
        })
        tag = "rerank" if use_rerank else "no rr "
        print(f"  {tag} | {q[:55]:55s} | refused={result['refused_at']}")

print(f"\nCompleted {len(rerank_runs)} runs in {time.time()-t0:.0f}s")
with open(SWEEP_DIR / "rerank_runs.pkl", "wb") as f:
    pickle.dump(rerank_runs, f)
print("Saved: rerank_runs.pkl")

  no rr  | What trials use pembrolizumab in non-small cell lung ca | refused=None
  rerank | What trials use pembrolizumab in non-small cell lung ca | refused=None
  no rr  | Which trials target the BRAF V600E mutation?            | refused=None
  rerank | Which trials target the BRAF V600E mutation?            | refused=None
  no rr  | Are there trials studying antibody drug conjugates for  | refused=generation
  rerank | Are there trials studying antibody drug conjugates for  | refused=None
  no rr  | What are reported hazard ratios for overall survival in | refused=generation
  rerank | What are reported hazard ratios for overall survival in | refused=generation
  no rr  | What are the eligibility criteria for trials enrolling  | refused=generation
  rerank | What are the eligibility criteria for trials enrolling  | refused=None
  no rr  | What trials use CAR-T cell therapy for leukemia or lymp | refused=generation
  rerank | What trials use CAR-T cell therapy for leukemia or lymp |

In [13]:
# Cell D3 — score the 12 reranker runs with LLM judge (faithfulness + context precision)
# ~5 min, ~20K Groq tokens. Reuses helpers from Cell M3.

print(f"Scoring {len(rerank_runs)} runs (~5 min, ~20K Groq tokens)\n")

rerank_scored = []
t0 = time.time()
for i, run in enumerate(rerank_runs):
    ctx_text = "\n\n---\n\n".join(run["contexts"])
    f_score, n_claims, n_sup = faithfulness_score(run["answer"], ctx_text)
    prec = context_precision_score(run["query"], run["contexts"])
    rerank_scored.append({
        **run,
        "faithfulness":      round(f_score, 3),
        "n_claims":          n_claims,
        "n_supported":       n_sup,
        "context_precision": round(prec, 3),
    })
    elapsed = time.time() - t0
    tag = "rerank" if run["use_reranker"] else "no rr "
    print(f"  [{i+1:2d}/12] {tag} "
          f"faith={f_score:.2f} ({n_sup}/{n_claims})  "
          f"prec={prec:.2f}  "
          f"elapsed={elapsed:.0f}s  | {run['query'][:50]}")

print(f"\nDone in {time.time()-t0:.0f}s")
with open(SWEEP_DIR / "rerank_scored.pkl", "wb") as f:
    pickle.dump(rerank_scored, f)
print("Saved: rerank_scored.pkl")

Scoring 12 runs (~5 min, ~20K Groq tokens)

  [ 1/12] no rr  faith=0.62 (5/8)  prec=0.40  elapsed=7s  | What trials use pembrolizumab in non-small cell lu
    rate-limited, sleeping 5s
    rate-limited, sleeping 10s
    judge failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqn8easje2bsv9wj4fmmhcwx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99675, Requested 1012. Please try again in 9m53.568s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
    rate-limited, sleeping 5s
    rate-limited, sleeping 10s
    judge failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqn8easje2bsv9wj4fmmhcwx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99657, Requested 1013. Please try again in 9m38.88s. Need more t

In [14]:
# Cell D4 — summarize rerank on vs off

df_rr = pd.DataFrame(rerank_scored)
df_rr["was_refused"] = df_rr["answer"].apply(is_refusal)

# Aggregate summary
rows = []
for use_rr in [False, True]:
    sub = df_rr[df_rr["use_reranker"] == use_rr]
    answered = sub[~sub["was_refused"]]
    rows.append({
        "config":                  "with reranker" if use_rr else "no reranker",
        "n_total":                 len(sub),
        "n_answered":              len(answered),
        "n_refused":               int(sub["was_refused"].sum()),
        "refusal_rate":            round(sub["was_refused"].mean(), 3),
        "faith_when_answered":     round(answered["faithfulness"].mean(), 3) if len(answered) else None,
        "ctx_precision_all":       round(sub["context_precision"].mean(), 3),
        "ctx_precision_answered":  round(answered["context_precision"].mean(), 3) if len(answered) else None,
    })
rerank_summary = pd.DataFrame(rows).set_index("config")

print("=" * 80)
print("RERANKER COMPARISON (chunk size = 1000, 6 in-corpus queries × 2 configs)")
print("=" * 80)
print(rerank_summary)

# Per-query refusal delta
print("\n" + "=" * 80)
print("PER-QUERY REFUSAL OUTCOMES")
print("=" * 80)
pivot = df_rr.pivot_table(
    index="query",
    columns="use_reranker",
    values="was_refused",
    aggfunc="first",
).rename(columns={False: "no_rr_refused", True: "rerank_refused"})

def delta_label(r):
    if r["no_rr_refused"] and not r["rerank_refused"]: return "FIXED by rerank"
    if r["no_rr_refused"] and r["rerank_refused"]:     return "still refused"
    if not r["no_rr_refused"] and not r["rerank_refused"]: return "still answered"
    return "newly refused (regression)"

pivot["delta"] = pivot.apply(delta_label, axis=1)
print(pivot)

# Per-query context precision delta
print("\n" + "=" * 80)
print("PER-QUERY CONTEXT PRECISION")
print("=" * 80)
prec_pivot = df_rr.pivot_table(
    index="query",
    columns="use_reranker",
    values="context_precision",
    aggfunc="first",
).rename(columns={False: "no_rr", True: "rerank"})
prec_pivot["delta"] = (prec_pivot["rerank"] - prec_pivot["no_rr"]).round(3)
print(prec_pivot)

# Save artifacts
rerank_summary.to_csv(SWEEP_DIR / "rerank_summary.csv")
pivot.to_csv(SWEEP_DIR / "rerank_per_query_refusal.csv")
prec_pivot.to_csv(SWEEP_DIR / "rerank_per_query_precision.csv")
print("\nSaved: rerank_summary.csv, rerank_per_query_refusal.csv, rerank_per_query_precision.csv")

RERANKER COMPARISON (chunk size = 1000, 6 in-corpus queries × 2 configs)
               n_total  n_answered  n_refused  refusal_rate  \
config                                                        
no reranker          6           2          4         0.667   
with reranker        6           4          2         0.333   

               faith_when_answered  ctx_precision_all  ctx_precision_answered  
config                                                                         
no reranker                  0.812              0.067                    0.20  
with reranker                0.812              0.100                    0.15  

PER-QUERY REFUSAL OUTCOMES
use_reranker                                        no_rr_refused  \
query                                                               
Are there trials studying antibody drug conjuga...           True   
What are reported hazard ratios for overall sur...           True   
What are the eligibility criteria for trials en...